# Exercise 03 — MovieLens centrality analysis

This notebook builds on Exercise 02 and uses the same MovieLens `ml-latest-small` bipartite graph. It computes Lecture 03 metrics for node importance and structural interpretation, including degree, PageRank, betweenness, closeness, density, average clustering, and path-based metrics.

## Setup

Import the packages used for graph analysis and visualization.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from statistics import mean

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
pd.options.display.max_rows = 20

## Load the same graph

Load `ratings.csv` and `movies.csv` and build the same bipartite user–movie graph used in Exercise 02.

In [ ]:
ratings = pd.read_csv('../movielense/ml-latest-small/ratings.csv')
movies = pd.read_csv('../movielense/ml-latest-small/movies.csv')

ratings['user_node'] = ratings['userId'].astype(str).radd('user_')
ratings['movie_node'] = ratings['movieId'].astype(str).radd('movie_')

G = nx.Graph()
G.add_nodes_from([(u, {'bipartite': 0, 'node_type': 'user'}) for u in ratings['user_node'].unique()])
G.add_nodes_from([(m, {'bipartite': 1, 'node_type': 'movie'}) for m in ratings['movie_node'].unique()])
G.add_edges_from([(row.user_node, row.movie_node) for row in ratings.itertuples(index=False)])

print('Loaded graph with', G.number_of_nodes(), 'nodes and', G.number_of_edges(), 'edges')
print('Connected:', nx.is_connected(G))

## Graph metrics

Compute network-wide structure metrics for the MovieLens graph.

In [ ]:
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G)
avg_clustering = nx.average_clustering(G)
diameter = nx.diameter(G)
avg_shortest = nx.average_shortest_path_length(G)

print('Total nodes:', num_nodes)
print('Total edges:', num_edges)
print('Density:', f'{density:.6f}')
print('Average clustering:', f'{avg_clustering:.6f}')
print('Diameter:', diameter)
print('Average shortest path length:', f'{avg_shortest:.6f}')

## Centrality measures

Compute several centrality scores for every node.

In [ ]:
degree_c = nx.degree_centrality(G)
pagerank_c = nx.pagerank(G, alpha=0.85, max_iter=200)
betweenness_c = nx.betweenness_centrality(G, normalized=True)
closeness_c = nx.closeness_centrality(G)

centrality = pd.DataFrame({
    'degree': pd.Series(degree_c),
    'pagerank': pd.Series(pagerank_c),
    'betweenness': pd.Series(betweenness_c),
    'closeness': pd.Series(closeness_c)
})
centrality['node_type'] = centrality.index.map(lambda n: G.nodes[n]['node_type'])
centrality.sort_values('pagerank', ascending=False).head(10)

## Top 5 nodes by each centrality measure

Compare the highest-ranking nodes across degree, PageRank, betweenness, and closeness.

In [ ]:
top_degree = centrality.sort_values('degree', ascending=False).head(5)
top_pagerank = centrality.sort_values('pagerank', ascending=False).head(5)
top_betweenness = centrality.sort_values('betweenness', ascending=False).head(5)
top_closeness = centrality.sort_values('closeness', ascending=False).head(5)

print('Top 5 by degree centrality:')
display(top_degree[['node_type', 'degree']])
print('Top 5 by PageRank:')
display(top_pagerank[['node_type', 'pagerank']])
print('Top 5 by betweenness centrality:')
display(top_betweenness[['node_type', 'betweenness']])
print('Top 5 by closeness centrality:')
display(top_closeness[['node_type', 'closeness']])

## Visualization of central nodes

Draw a small annotated subgraph with node size reflecting degree and node color reflecting PageRank.

In [ ]:
top_nodes = list(set(top_pagerank.index[:10]) | set(top_degree.index[:10]) | set(top_betweenness.index[:10]))
sample_subgraph = G.subgraph(top_nodes).copy()
pos = nx.spring_layout(sample_subgraph, seed=42)
sizes = [500 + 3000 * degree_c[node] for node in sample_subgraph]
colors = [pagerank_c[node] for node in sample_subgraph]
labels = {node: node.replace('user_', 'U').replace('movie_', 'M') for node in sample_subgraph}

plt.figure(figsize=(10, 8))
nx.draw_networkx_nodes(sample_subgraph, pos, node_size=sizes, node_color=colors, cmap='plasma')
nx.draw_networkx_edges(sample_subgraph, pos, alpha=0.6)
nx.draw_networkx_labels(sample_subgraph, pos, labels=labels, font_size=10)
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(vmin=min(colors), vmax=max(colors)))
sm.set_array([])
plt.colorbar(sm, label='PageRank', ax=plt.gca())
plt.title('Annotated subgraph: node size ~ degree centrality, color ~ PageRank')
plt.axis('off')
plt.show()

## Interpretation

Summarize what the centrality measures reveal about the MovieLens bipartite network.

- Degree centrality highlights users with the largest number of ratings, which are naturally the most connected nodes in this bipartite graph.
- PageRank largely agrees with degree in this dataset, since the highest-rated users are connected to many movies and also to popular movie nodes.
- Betweenness centrality can identify users or movies that lie on many shortest paths, acting as structural bridges between otherwise separate parts of the graph.
- Closeness centrality shows which nodes have the shortest average distance to all others, which is especially meaningful in a connected bipartite network.

Different metrics do not always agree because degree counts direct connections, while PageRank values neighbors' importance, and betweenness emphasizes nodes that link different regions of the network. In a user–movie graph, the most active users tend to dominate degree-based rankings, but some movies or users with strong bridging roles may rise in betweenness or closeness even if they have fewer direct edges.